# Section 3: Cloud Root Analysis

**3.2**  Cloud-root cross-sections in non-dimensionalised coordinates (z/z_sl, x/L)

> **Section 3.2 requires 3D data.** From the run directory, run:
> ```
> cd $RUN_DIR
> python $MICROHH/python/3d_to_nc.py -v thl qt ql w b
> ```
> then set `HAS_3D = True` below.

**Section 3.3 requires preprocessed composite files.**  Run (after experiments finish):
```bash
# Step 1: convert binary dumps to NetCDF (once per run dir)
bash analysis/submit_3d_to_nc.sh --expt base
# Step 2: extract composite events (one SLURM job per rep)
bash analysis/cloud_root_composite_submit.sh --expts base
# Step 3: average across reps (runs in seconds, can run interactively)
python analysis/cloud_root_composite_average.py --expt base --rt 2stream \
    --composite-dir $SCRATCH/CASS_LES/analysis/cloud_root_composite
```
For **debug runs**: `python analysis/cloud_root_composite_debug.py --rt 2stream`

In [1]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from cass_analysis import (
    RunSet,
    load_stats_ensemble,
    load_xy_files,
    load_3d_nc,
    compute_z_sl,
    compute_normalized_cloud_root_profiles,
    plot_normalized_cloud_root_profiles,
    RT_LABEL,
)

# ── CONFIGURATION ─────────────────────────────────────────────────────────────
SCRATCH = "/pscratch/sd/m/mpowell/CASS_LES"
EXPT    = "experiments/no_aerosols_zero_wind"      # change to switch experiment
N_REPS  = 4              # 1 for debug, 4 for production
HAS_3D  = True             # set True after running 3d_to_nc.py
# ─────────────────────────────────────────────────────────────────────────────

rs = RunSet(EXPT, f"{SCRATCH}/{EXPT}", n_reps=N_REPS)
print(rs)


RunSet('experiments/no_aerosols_zero_wind', root=/pscratch/sd/m/mpowell/CASS_LES/experiments/no_aerosols_zero_wind)
  2stream: 1 rep(s)
  raytracer: 1 rep(s)


In [2]:
# Load stats for both RT types
s2s_mean, s2s_std = load_stats_ensemble(rs.dirs["2stream"])
srt_mean, srt_std = load_stats_ensemble(rs.dirs["raytracer"])

# Estimate z_sl at the end of the simulation (mature cloud period)
z_sl_2s = compute_z_sl(s2s_mean, time_idx=-1)
z_sl_rt = compute_z_sl(srt_mean, time_idx=-1)
print(f"z_sl (2stream):   {z_sl_2s:.0f} m")
print(f"z_sl (raytracer): {z_sl_rt:.0f} m")

z_sl (2stream):   1538 m
z_sl (raytracer): 1562 m


## 3.3  L&P 2014 composite cloud-root cross-sections  (2stream vs raytracer)

All cloud objects with **chord length ≥ 1 km** in each slice direction, from all dump times
between **11:30–15:30 LST**, are collected and composited following Lohou & Patton (2014):

1. For each qualifying object, extract an xz-slice (y = centroid row) **and** a yz-slice (x = centroid column)
2. Scale the horizontal axis: $x \to x/L$, where $L$ is the chord length of the cloud mask in that slice direction
3. Shift so the cloud centroid is at $x/L = 0$
4. Scale the vertical axis: $z \to z/z_\text{sl}(t)$ using the subcloud-layer height at that dump time
5. Bilinear-interpolate each event onto a standard $[-1, 1] \times [0, 1.5]$ grid
6. Average: per-rep first, then across reps — so each rep contributes equally

In [3]:
from cass_analysis import (
    load_composite_events, composite_mean_std,
    plot_cloud_root_composite,
    XL_GRID, ZND_GRID,
)
import xarray as xr

# ── CONFIGURATION ─────────────────────────────────────────────────────────────
COMPOSITE_ROOT = f"{SCRATCH}/analysis/cloud_root_composite"
# Uses EXPT from the top config cell — change EXPT there to switch experiment
# ─────────────────────────────────────────────────────────────────────────────

_IS_DEBUG = EXPT.startswith("debug/")

def _load_composite(composite_root, expt, rt, orient):
    """Load composite dataset.

    For production experiments:
      Loads composite_{orient}.nc — produced by cloud_root_composite_average.py.
      Raises FileNotFoundError with a clear message if it does not exist.

    For debug experiments (EXPT starts with "debug/"):
      Falls back to events_{orient}.nc in rep_01/ and computes the mean on-the-fly.
    """
    base = f"{composite_root}/{expt}/{rt}"
    avg_path = f"{base}/composite_{orient}.nc"
    try:
        ds = xr.open_dataset(avg_path)
        print(f"  {rt}/{orient}: averaged composite  (n_reps={int(ds.get('n_reps', '?'))})")
        return ds
    except FileNotFoundError:
        pass

    if _IS_DEBUG:
        # Debug fallback: compute composite on-the-fly from rep_01 events
        ev_path = f"{base}/rep_01/events_{orient}.nc"
        try:
            ds_ev = load_composite_events(ev_path)
            ds    = composite_mean_std(ds_ev)
            print(f"  {rt}/{orient}: debug events rep_01 only (n={ds_ev.sizes['event']})")
            return ds
        except FileNotFoundError:
            print(f"  {rt}/{orient}: NOT FOUND — run cloud_root_composite_debug.py first")
            return None
    else:
        print(f"  {rt}/{orient}: composite_{orient}.nc not found.")
        print(f"    → Run cloud_root_composite_submit.sh + cloud_root_composite_average.py first")
        print(f"    → Expected: {avg_path}")
        return None

comp_data = {}   # {rt: {orient: ds_composite | None}}
_mode = "debug" if _IS_DEBUG else "production"
print(f"Loading composites for EXPT='{EXPT}' ({_mode})")
for rt in ("2stream", "raytracer"):
    comp_data[rt] = {
        orient: _load_composite(COMPOSITE_ROOT, EXPT, rt, orient)
        for orient in ("xz", "yz")
    }


Loading composites for EXPT='experiments/no_aerosols_zero_wind' (production)
  2stream/xz: composite_xz.nc not found.
    → Run cloud_root_composite_submit.sh + cloud_root_composite_average.py first
    → Expected: /pscratch/sd/m/mpowell/CASS_LES/analysis/cloud_root_composite/experiments/no_aerosols_zero_wind/2stream/composite_xz.nc
  2stream/yz: composite_yz.nc not found.
    → Run cloud_root_composite_submit.sh + cloud_root_composite_average.py first
    → Expected: /pscratch/sd/m/mpowell/CASS_LES/analysis/cloud_root_composite/experiments/no_aerosols_zero_wind/2stream/composite_yz.nc
  raytracer/xz: composite_xz.nc not found.
    → Run cloud_root_composite_submit.sh + cloud_root_composite_average.py first
    → Expected: /pscratch/sd/m/mpowell/CASS_LES/analysis/cloud_root_composite/experiments/no_aerosols_zero_wind/raytracer/composite_xz.nc
  raytracer/yz: composite_yz.nc not found.
    → Run cloud_root_composite_submit.sh + cloud_root_composite_average.py first
    → Expected: /pscr

In [4]:
# ── 3.3a  2-stream vs raytracer composite (xz-slices) ────────────────────────
ORIENT = "xz"   # switch to "yz" to inspect the perpendicular composite

_available = {rt: comp_data[rt].get(ORIENT) for rt in ("2stream", "raytracer")}
_available = {k: v for k, v in _available.items() if v is not None}

if not _available:
    print(f"No composite data available for orientation '{ORIENT}'.")
    print("Run the prep and average steps first — see the markdown cell above.")
else:
    # Shared colour limits across both RT types
    def _collect(vn):
        arrs = [_available[rt][vn].values for rt in _available if vn in _available[rt]]
        return np.concatenate([a.ravel() for a in arrs])

    vmax_thl = float(np.nanpercentile(np.abs(_collect("w_thl_mean")), 98))
    vmax_qv  = float(np.nanpercentile(np.abs(_collect("w_qv_mean")),  98))
    vlim_thl = (-vmax_thl, vmax_thl)
    vlim_qv  = (-vmax_qv,  vmax_qv)

    fig, axes = plt.subplots(2, 2, figsize=(13, 10))

    for row, rt in enumerate(("2stream", "raytracer")):
        ax_l, ax_r = axes[row]
        if rt not in _available:
            ax_l.set_visible(False); ax_r.set_visible(False); continue
        plot_cloud_root_composite(ax_l, ax_r, _available[rt],
                                  vlim_thl=vlim_thl, vlim_qv=vlim_qv,
                                  label=f"{RT_LABEL[rt]}")

    plt.suptitle(
        f"{rs.name}  —  Composite cloud-root cross-sections  ({ORIENT})  "
        f"[11:30–15:30 LST,  chord ≥ 1 km]",
        y=1.01,
    )
    plt.tight_layout()
    plt.show()

No composite data available for orientation 'xz'.
Run the prep and average steps first — see the markdown cell above.


In [5]:
# ── 3.3b  Difference composite: raytracer − 2stream ──────────────────────────
# Shows where 3D RT shifts the composite turbulent structure
if (_available.get("2stream") is not None
        and _available.get("raytracer") is not None):

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    ax_l, ax_r = axes

    for var, ax, cmap, label in [
        ("w_thl_mean", ax_l, "RdBu_r", "Δw'θ_l'  (K m s⁻¹)"),
        ("w_qv_mean",  ax_r, "BrBG",   "Δw'q_v'  (g kg⁻¹ m s⁻¹)"),
    ]:
        diff = (_available["raytracer"][var].values
                - _available["2stream"][var].values)
        vmax = float(np.nanpercentile(np.abs(diff), 98))
        xL   = _available["2stream"].xL.values
        z_nd = _available["2stream"].z_nd.values
        pcm  = ax.pcolormesh(xL, z_nd, diff, cmap=cmap, shading="auto",
                             vmin=-vmax, vmax=vmax)
        plt.colorbar(pcm, ax=ax, label=label)
        ax.axvline(-0.5, color="0.4", lw=0.8, ls="--")
        ax.axvline( 0.5, color="0.4", lw=0.8, ls="--")
        ax.axhline( 1.0, color="0.4", lw=0.8, ls=":")
        ax.set_xlabel("x/L");  ax.set_ylabel("z / z_sl")
        ax.set_xlim(-1, 1);    ax.set_ylim(0, 1.0)
        ax.set_title(f"3D − 1D  {label.split('(')[0].strip()}")

    plt.suptitle(
        f"{rs.name}  —  Composite difference (raytracer − 2stream)  "
        f"[{ORIENT}, 11:30–15:30 LST]",
        y=1.01,
    )
    plt.tight_layout()
    plt.show()
else:
    print("Need both 2stream and raytracer composites to compute difference.")

Need both 2stream and raytracer composites to compute difference.
